[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/095.%20The%20Supply%20Chain%20Network%20Design%20under%20Uncertainty/P95-Tier-7_executed.ipynb)

---


# 95. Supply Chain Network Design under Uncertainty
## Tier 7: Human–AI Symbiotic Partnership

### Purpose and scope

This notebook builds a small, illustrative simulation of collaborative network-design decision-making. An AI component generates candidate facility networks, estimates their performance, and explains selected characteristics. A panel of human-expert agents evaluates a subset of candidates. Their ratings and AI confidence are combined into a consensus ranking, while feedback updates stored preference weights and trust scores.

The notebook is a decision-process simulation, not an exact or heuristic facility-location optimizer. Candidate networks are generated randomly; they are not guaranteed to satisfy capacity constraints or to be optimal. The model does not estimate real business outcomes, and its simulated expert ratings are not empirical human judgments.

### Inputs and outputs

Inputs include candidate facilities, customer zones, product categories, facility fixed costs and capacities, scenario probabilities and demand, and illustrative expert preferences. Outputs include candidate-level cost, utilization, service and risk indicators; explanations and confidence scores; expert feedback; consensus rankings; and plots of collaboration indicators.

### Modeling assumptions

- Each candidate opens between three and seven facilities. Every customer is assigned to one of the three geographically nearest open facilities, chosen randomly. These rules generate alternatives; they do not enforce capacity feasibility.
- Scenario demand is represented by a customer-level total demand, divided equally among products. Transportation costs vary by facility, customer, and product, but not by scenario. Expected variable cost and expected facility load use the scenario probabilities.
- Facility identifiers and customer zones are arranged so that `customer // 2` serves as a simple location proxy. This is an illustrative distance measure, not a geographic map.
- The service-level and risk indicators are deliberately simplified proxies. They are not calibrated probabilities of service or operational failure.
- Expert ratings, AI confidence, strategic alignment, preference updates, and trust updates are constructed for teaching purposes. In particular, the feedback trend is descriptive; it does not establish that recommendations improve because of feedback.
- The model omits facility-opening constraints, capacity-feasibility constraints, inventory, lead times, emissions, and monetary calibration. Costs are illustrative cost units.

### Collaboration workflow

1. Generate candidate networks and calculate performance indicators.
2. Explain each candidate and estimate recommendation confidence.
3. Have each expert assess the three lowest-cost candidates.
4. Update stored preference weights from the feedback and rank alternatives using trust-weighted expert ratings plus an AI-confidence contribution.
5. Record trust and agreement indicators, inspect the results, and discuss their limitations.

### Scoring logic

For candidate \(a\), let \(E_a\) be the experts who evaluated it, \(r_{ea}\) the rating from expert \(e\), \(t_e\) that expert's trust weight, and \(c_a\) the AI confidence on a 0-to-1 scale. The consensus score used in the code is

$$S_a = \sum_{e \in E_a} t_e r_{ea} + 0.3(10c_a).$$

The first term gives greater influence to ratings from experts with higher trust weights. The second term gives the AI-confidence contribution a fixed weight of 0.3 after converting confidence to a 0-to-10 scale. Candidates that experts did not discuss receive no expert-rating contribution; their score still includes the AI-confidence contribution. This score is a ranking rule for this simulation, not an objective function that proves a network is optimal.

The agreement indicator for one candidate is defined as \(1 - \operatorname{sd}(r_{ea})/10\), with a default of 0.5 when no expert rated that candidate. The session-level agreement indicator is based on the rating spread for alternatives assessed by at least two experts. These are descriptive measures, not formal tests of inter-rater reliability.

### What to examine

Inspect how explanations connect candidate choices to calculated indicators, how expert ratings and trust weights affect the ranking, and how the indicators vary across sessions. The plots describe this simulated process; they do not validate its assumptions or demonstrate real-world decision quality.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

# A fixed seed makes the generated instance and simulated judgments reproducible.
np.random.seed(42)
random.seed(42)

## 1. Define the instance

The instance collects the sets and parameters used by later steps. Scenario probabilities describe how much weight each demand realization receives in expected-value calculations. They must be nonnegative and sum to one. Keeping the instance explicit makes the data used by the AI component visible and makes its calculations traceable.

In [ ]:
@dataclass
class HumanAICollaborationInstance:
    """Data for the illustrative human–AI network-design simulation."""

    facilities: List[int]
    customers: List[int]
    products: List[int]
    scenarios: List[int]
    fixed_costs: List[float]
    capacities: List[float]
    transport_costs: Dict[Tuple[int, int, int], float]
    scenario_probs: List[float]
    scenario_demands: Dict[Tuple[int, int], float]
    human_experts: List[str]
    strategic_factors: Dict[str, float]

    def __post_init__(self) -> None:
        if len(self.fixed_costs) != len(self.facilities):
            raise ValueError("Each facility must have one fixed cost.")
        if len(self.capacities) != len(self.facilities):
            raise ValueError("Each facility must have one capacity.")
        if len(self.scenario_probs) != len(self.scenarios):
            raise ValueError("Each scenario must have one probability.")
        if any(probability < 0 for probability in self.scenario_probs):
            raise ValueError("Scenario probabilities cannot be negative.")
        if not np.isclose(sum(self.scenario_probs), 1.0):
            raise ValueError("Scenario probabilities must sum to one.")


def create_human_ai_instance() -> HumanAICollaborationInstance:
    """Create the illustrative facilities, costs, scenarios, and expert factors."""
    facilities = list(range(12))
    customers = list(range(20))
    products = list(range(5))
    scenarios = list(range(8))

    fixed_costs = [
        180, 220, 160, 250, 200, 170,
        210, 190, 240, 150, 205, 215,
    ]
    capacities = [
        150, 180, 130, 200, 160, 140,
        170, 150, 190, 120, 155, 165,
    ]

    transport_costs = {}
    for facility in facilities:
        for customer in customers:
            for product in products:
                distance_factor = abs(facility - customer // 2) * 0.8
                product_factor = 1.0 + product * 0.15
                base_cost = 2.5 + distance_factor + product_factor
                transport_costs[(facility, customer, product)] = max(
                    1.8,
                    base_cost + np.random.normal(0, 0.3),
                )

    scenario_probs = [0.15, 0.20, 0.25, 0.15, 0.10, 0.05, 0.05, 0.05]
    scenario_demands = {
        (scenario, customer): float(np.random.uniform(50, 150))
        for scenario in scenarios
        for customer in customers
    }

    human_experts = [
        "Supply Chain Strategist",
        "Regional Operations Manager",
        "Financial Analyst",
        "Risk Management Specialist",
        "Market Intelligence Expert",
    ]
    strategic_factors = {
        "government_stability": 0.8,
        "market_growth_potential": 0.9,
        "competitive_positioning": 0.7,
        "infrastructure_quality": 0.85,
        "labor_availability": 0.75,
        "regulatory_environment": 0.7,
        "cultural_alignment": 0.8,
        "expansion_readiness": 0.85,
    }

    return HumanAICollaborationInstance(
        facilities=facilities,
        customers=customers,
        products=products,
        scenarios=scenarios,
        fixed_costs=fixed_costs,
        capacities=capacities,
        transport_costs=transport_costs,
        scenario_probs=scenario_probs,
        scenario_demands=scenario_demands,
        human_experts=human_experts,
        strategic_factors=strategic_factors,
    )


human_ai_instance = create_human_ai_instance()
print("Human–AI collaboration instance")
print(f"Facilities: {len(human_ai_instance.facilities)}")
print(f"Customers: {len(human_ai_instance.customers)}")
print(f"Products: {len(human_ai_instance.products)}")
print(f"Demand scenarios: {len(human_ai_instance.scenarios)}")
print(f"Human experts: {len(human_ai_instance.human_experts)}")
print(f"Scenario probability total: {sum(human_ai_instance.scenario_probs):.2f}")
print("Strategic-factor scores:")
for factor, score in human_ai_instance.strategic_factors.items():
    print(f"  {factor}: {score:.2f}")

## 2. Generate and explain AI alternatives

An alternative is a set of open facilities together with one assigned facility for every customer. The generator uses a local random-number generator for each alternative so that the candidate layouts are reproducible without resetting the notebook's global random state.

For a candidate \(a\), expected variable cost is calculated across demand scenarios:

$$C_a^{\mathrm{var}} = \sum_{s \in S} p_s \sum_{c \in C} \sum_{p \in P} d_{sc}/|P| \; q_{f(c),c,p},$$

where \(S\) is the scenario set, \(C\) the customer set, \(P\) the product set, \(p_s\) the probability of scenario \(s\), \(d_{sc}\) the total demand of customer \(c\) in scenario \(s\), \(f(c)\) the facility assigned to that customer, and \(q_{f(c),c,p}\) the corresponding unit transportation cost. The code adds the fixed costs of the open facilities to obtain expected total cost.

Expected facility load is calculated using the same scenario probabilities. Utilization is expected load divided by capacity; values above one indicate that expected load exceeds the stated capacity. Because capacity is not enforced by the candidate generator, such an alternative remains a candidate for discussion, not a capacity-feasible plan. Service and risk scores are transparent illustrative proxies defined in the code.

In [ ]:
class ExplainableAI:
    """Generate candidate networks, indicators, explanations, and confidence scores."""

    def __init__(self, instance: HumanAICollaborationInstance):
        self.instance = instance
        self.confidence_threshold = 0.7
        self.explainability_level = "detailed"
        self.human_feedback_history = []
        self.preference_weights = {
            "cost": 0.3,
            "risk": 0.2,
            "service": 0.2,
            "strategic": 0.3,
        }

    def generate_network_alternatives(self, num_alternatives: int = 10) -> List[Dict]:
        """Create alternatives and attach their metrics, explanations, and confidence."""
        alternatives = []
        for alternative_id in range(num_alternatives):
            config = self._generate_network_configuration(alternative_id)
            metrics = self._calculate_metrics(config)
            explanation = self._generate_explanation(config, metrics, alternative_id)
            confidence = self._calculate_confidence(config, metrics, alternative_id)
            alternatives.append({
                "id": alternative_id,
                "configuration": config,
                "metrics": metrics,
                "explanation": explanation,
                "confidence": confidence,
                "strategic_alignment": self._calculate_strategic_alignment(config),
            })
        return alternatives

    def _generate_network_configuration(self, seed: int) -> Dict:
        """Choose open facilities and assign each customer to a nearby candidate."""
        rng = np.random.RandomState(seed + 100)
        num_open = int(rng.randint(3, 8))
        open_facilities = rng.choice(
            self.instance.facilities, num_open, replace=False
        )
        assignments = {}
        for customer in self.instance.customers:
            ranked = sorted(
                (abs(facility - customer // 2), facility)
                for facility in open_facilities
            )
            nearest_facilities = [facility for _, facility in ranked[:3]]
            assignments[customer] = int(rng.choice(nearest_facilities))
        return {
            "open_facilities": set(int(facility) for facility in open_facilities),
            "customer_assignments": assignments,
        }

    def _calculate_metrics(self, config: Dict) -> Dict:
        """Calculate expected cost, utilization, and illustrative service and risk scores."""
        open_facilities = config["open_facilities"]
        assignments = config["customer_assignments"]
        fixed_cost = sum(self.instance.fixed_costs[f] for f in open_facilities)
        expected_variable_cost = 0.0
        expected_loads = {facility: 0.0 for facility in open_facilities}

        for scenario, probability in zip(
            self.instance.scenarios, self.instance.scenario_probs
        ):
            for customer in self.instance.customers:
                facility = assignments[customer]
                demand = self.instance.scenario_demands[(scenario, customer)]
                expected_loads[facility] += probability * demand
                product_demand = demand / len(self.instance.products)
                for product in self.instance.products:
                    unit_cost = self.instance.transport_costs[
                        (facility, customer, product)
                    ]
                    expected_variable_cost += (
                        probability * product_demand * unit_cost
                    )

        expected_cost = fixed_cost + expected_variable_cost
        utilizations = {
            facility: expected_loads[facility] / self.instance.capacities[facility]
            for facility in open_facilities
        }
        utilization_values = list(utilizations.values())
        avg_utilization = float(np.mean(utilization_values))
        max_utilization = float(max(utilization_values))
        utilization_variance = float(np.var(utilization_values))

        # These formulas are teaching proxies, not calibrated operational measures.
        service_level = max(0.0, min(1.0, 1.0 - utilization_variance * 2))
        risk_score = utilization_variance
        if max_utilization > 0.8:
            risk_score += (max_utilization - 0.8) * 2

        return {
            "fixed_cost": fixed_cost,
            "variable_cost": expected_variable_cost,
            "expected_cost": expected_cost,
            "avg_utilization": avg_utilization,
            "max_utilization": max_utilization,
            "utilization_variance": utilization_variance,
            "facility_utilizations": utilizations,
            "service_level": service_level,
            "risk_score": risk_score,
            "num_facilities": len(open_facilities),
        }

    def _generate_explanation(self, config: Dict, metrics: Dict, alt_id: int) -> Dict:
        """Summarize candidate choices and connect them to calculated indicators."""
        explanation = {
            "summary": (
                f"Candidate {alt_id} opens {metrics['num_facilities']} facilities."
            ),
            "key_decisions": [],
            "performance_rationale": [],
            "trade_offs": [],
            "risk_factors": [],
            "recommendations": [],
        }
        open_facilities = sorted(config["open_facilities"])
        explanation["key_decisions"].append(
            f"Open facilities: {open_facilities}. Each customer is assigned to one open facility."
        )

        if metrics["expected_cost"] < 5000:
            explanation["performance_rationale"].append(
                "Expected cost is below the illustrative 5,000-cost-unit reference."
            )
        if metrics["service_level"] > 0.9:
            explanation["performance_rationale"].append(
                "The utilization-variance proxy gives a high service indicator."
            )
        if metrics["num_facilities"] > 5:
            explanation["trade_offs"].append(
                "More facilities raise fixed cost, while offering more locations for assignments."
            )
        else:
            explanation["trade_offs"].append(
                "Fewer facilities reduce fixed cost, while potentially increasing transport cost or load."
            )
        if metrics["max_utilization"] > 0.9:
            explanation["trade_offs"].append(
                "At least one facility has high expected utilization; capacity feasibility needs review."
            )
        if metrics["risk_score"] > 0.1:
            explanation["risk_factors"].append(
                f"The illustrative risk score is {metrics['risk_score']:.3f}."
            )
        if metrics["utilization_variance"] > 0.05:
            explanation["risk_factors"].append(
                "Uneven expected utilization may indicate potential bottlenecks."
            )
        if metrics["service_level"] < 0.85:
            explanation["recommendations"].append(
                "Review customer assignments and capacity before using this candidate."
            )
        if metrics["max_utilization"] > 0.95:
            explanation["recommendations"].append(
                "Investigate capacity expansion or demand redistribution."
            )
        return explanation

    def _calculate_confidence(
        self, config: Dict, metrics: Dict, alternative_id: int
    ) -> float:
        """Create a reproducible illustrative confidence score from simple penalties."""
        quality_score = 1.0
        if metrics["risk_score"] > 0.1:
            quality_score -= 0.2
        if metrics["service_level"] < 0.8:
            quality_score -= 0.2
        if metrics["max_utilization"] > 0.95 or metrics["avg_utilization"] < 0.3:
            quality_score -= 0.1
        rng = np.random.RandomState(alternative_id + 200)
        confidence = quality_score + rng.normal(0, 0.1)
        return float(max(0.3, min(0.95, confidence)))

    def _calculate_strategic_alignment(self, config: Dict) -> float:
        """Calculate a simple alignment proxy from the facility-region grouping."""
        open_facilities = config["open_facilities"]
        high_growth_count = sum(facility < 6 for facility in open_facilities)
        stable_count = len(open_facilities) - high_growth_count
        return (high_growth_count * 0.6 + stable_count * 0.4) / len(open_facilities)

    def incorporate_human_feedback(
        self,
        alternative_id: int,
        human_rating: int,
        human_comments: str,
        expert_focus: str,
    ) -> None:
        """Record a rating and update the stored preference weights."""
        if not 1 <= human_rating <= 10:
            raise ValueError("Human ratings must be integers from 1 to 10.")
        self.human_feedback_history.append({
            "alternative_id": alternative_id,
            "human_rating": human_rating,
            "human_comments": human_comments,
            "expert_focus": expert_focus,
        })
        self._update_preference_weights(expert_focus, human_rating)

    def _update_preference_weights(self, expert_focus: str, rating: int) -> None:
        """Adjust and renormalize the preference category associated with an expert."""
        focus_mapping = {
            "Supply Chain Strategist": "strategic",
            "Financial Analyst": "cost",
            "Risk Management Specialist": "risk",
            "Regional Operations Manager": "service",
            "Market Intelligence Expert": "strategic",
        }
        category = focus_mapping.get(expert_focus, "strategic")
        if rating > 7:
            self.preference_weights[category] = min(
                0.5, self.preference_weights[category] + 0.05
            )
        elif rating < 5:
            self.preference_weights[category] = max(
                0.1, self.preference_weights[category] - 0.05
            )
        total_weight = sum(self.preference_weights.values())
        for key in self.preference_weights:
            self.preference_weights[key] /= total_weight


print("Explainable AI component is ready.")

## 3. Represent human experts

Each expert agent applies simple thresholds to candidate cost, risk, service, strategic alignment, and AI confidence. The resulting integer rating is a simulated judgment, not a prediction of what a real person would say. An expert discusses the three candidates with the lowest expected cost, which makes the human review selective rather than exhaustive. The feedback is recorded under the expert's full name so that it updates the intended preference category.

In [ ]:
class HumanExpert:
    """Simulate one expert's ratings, discussion points, concerns, and suggestions."""

    def __init__(
        self, name: str, expertise: str, preference_weights: Dict[str, float]
    ):
        self.name = name
        self.expertise = expertise
        self.preference_weights = preference_weights
        self.risk_tolerance = float(np.random.uniform(0.3, 0.8))
        self.strategic_focus = float(np.random.uniform(0.6, 0.9))
        self.cost_sensitivity = float(np.random.uniform(0.4, 0.8))
        self.decision_history = []
        self.confidence_history = []

    def evaluate_alternative(self, alternative: Dict) -> Tuple[int, str]:
        """Rate one alternative using the agent's illustrative decision rules."""
        metrics = alternative["metrics"]
        explanation = alternative["explanation"]
        rating = 5.0
        if metrics["expected_cost"] < 4000:
            rating += self.cost_sensitivity * 2
        elif metrics["expected_cost"] > 6000:
            rating -= self.cost_sensitivity * 2
        if metrics["risk_score"] < 0.05:
            rating += (1 - self.risk_tolerance) * 2
        elif metrics["risk_score"] > 0.15:
            rating -= (1 - self.risk_tolerance) * 2
        if metrics["service_level"] > 0.9:
            rating += 1.5
        elif metrics["service_level"] < 0.8:
            rating -= 1.5
        rating += alternative["strategic_alignment"] * self.strategic_focus * 2
        rating += alternative["confidence"] * 0.5
        rating += np.random.normal(0, 0.5)
        rating = int(max(1, min(10, rating)))
        comments = self._generate_comments(rating, metrics, explanation)
        self.decision_history.append({
            "alternative_id": alternative["id"],
            "rating": rating,
            "comments": comments,
        })
        self.confidence_history.append(alternative["confidence"])
        return rating, comments

    def _generate_comments(self, rating: int, metrics: Dict, explanation: Dict) -> str:
        """Create concise comments linked to the rating and candidate indicators."""
        comments = []
        if rating >= 8:
            comments.append("Strong simulated rating.")
            if metrics["service_level"] > 0.9:
                comments.append("The service proxy is high.")
            if metrics["expected_cost"] < 4500:
                comments.append("Expected cost is relatively low under this model.")
            if self.strategic_focus > 0.7:
                comments.append("Strategic alignment is worth reviewing.")
        elif rating >= 6:
            comments.append("Positive rating with some considerations.")
            if metrics["risk_score"] > 0.1:
                comments.append("The illustrative risk indicator needs attention.")
            if metrics["max_utilization"] > 0.9:
                comments.append("Expected utilization is high at one or more facilities.")
        elif rating >= 4:
            comments.append("Mixed rating; review the performance trade-offs.")
            if metrics["service_level"] < 0.85:
                comments.append("The service proxy is relatively low.")
            if self.cost_sensitivity > 0.6 and metrics["expected_cost"] > 5000:
                comments.append("Expected cost is high for a cost-focused review.")
        else:
            comments.append("Low simulated rating; substantial review is warranted.")
            if metrics["risk_score"] > 0.15:
                comments.append("The illustrative risk indicator is high.")
            if metrics["service_level"] < 0.8:
                comments.append("The service proxy is low.")
        if "Financial" in self.name and rating < 6:
            comments.append("Review the cost assumptions and financial implications.")
        elif "Risk" in self.name and metrics["risk_score"] > 0.1:
            comments.append("Review possible risk-mitigation actions.")
        elif "Strategist" in self.name:
            comments.append("Consider long-term strategic implications.")
        return " ".join(comments)

    def participate_in_discussion(self, alternatives: List[Dict]) -> Dict:
        """Evaluate the three lowest-cost candidates and summarize discussion."""
        top_alternatives = sorted(
            alternatives, key=lambda item: item["metrics"]["expected_cost"]
        )[:3]
        discussion_points = []
        for alternative in top_alternatives:
            rating, comments = self.evaluate_alternative(alternative)
            discussion_points.append({
                "alternative_id": alternative["id"],
                "rating": rating,
                "comments": comments,
                "key_concerns": self._identify_concerns(alternative),
                "suggestions": self._generate_suggestions(alternative),
            })
        return {
            "expert": self.name,
            "expertise": self.expertise,
            "discussion_points": discussion_points,
            "overall_assessment": self._provide_overall_assessment(top_alternatives),
        }

    def _identify_concerns(self, alternative: Dict) -> List[str]:
        """Identify candidate indicators that warrant review."""
        concerns = []
        metrics = alternative["metrics"]
        if metrics["risk_score"] > 0.12:
            concerns.append(f"High illustrative risk score ({metrics['risk_score']:.3f})")
        if metrics["max_utilization"] > 0.95:
            concerns.append("Expected utilization exceeds 95% at a facility")
        if metrics["service_level"] < 0.85:
            concerns.append("Low service proxy")
        if alternative["confidence"] < 0.6:
            concerns.append("Low simulated recommendation confidence")
        return concerns

    def _generate_suggestions(self, alternative: Dict) -> List[str]:
        """Suggest checks or adjustments for the candidate under review."""
        suggestions = []
        metrics = alternative["metrics"]
        if metrics["utilization_variance"] > 0.08:
            suggestions.append("Review assignments for uneven expected utilization.")
        if metrics["max_utilization"] > 0.9:
            suggestions.append("Check facility capacity and possible demand redistribution.")
        if metrics["service_level"] < 0.9:
            suggestions.append("Review assignments and service assumptions.")
        if metrics["num_facilities"] < 4:
            suggestions.append("Assess whether more facilities would improve the trade-off.")
        return suggestions

    def _provide_overall_assessment(self, alternatives: List[Dict]) -> str:
        """Summarize average indicators for the candidates discussed by this expert."""
        avg_cost = np.mean([item["metrics"]["expected_cost"] for item in alternatives])
        avg_service = np.mean([item["metrics"]["service_level"] for item in alternatives])
        avg_risk = np.mean([item["metrics"]["risk_score"] for item in alternatives])
        assessment = (
            f"The discussed candidates average {avg_cost:.1f} cost units, "
            f"a service proxy of {avg_service:.3f}, and an illustrative risk score "
            f"of {avg_risk:.3f}. "
        )
        if self.cost_sensitivity > 0.6:
            assessment += "Give particular attention to cost assumptions. "
        if self.risk_tolerance < 0.5:
            assessment += "Review risk indicators carefully. "
        if self.strategic_focus > 0.7:
            assessment += "Consider longer-term strategic alignment."
        return assessment


print("Human expert agents are ready.")

## 4. Run collaborative sessions and build consensus

The session follows the stated sequence: the AI generates candidates, experts review their selected candidates, feedback updates the stored preference weights, and a consensus score ranks the alternatives. Trust weights begin equally and are adjusted using session agreement and mean AI confidence. In this simulation, preference weights are stored and updated for transparency but do not directly change candidate generation, expert ratings, or consensus scores. The feedback history therefore records interaction; it is not an adaptive optimization algorithm.

Consensus strength compares the highest score with the second-highest score. A larger gap means the ranking is less close under this scoring rule, but it does not mean that the selected candidate is objectively better or that experts agree about every decision.

In [ ]:
class HumanAICollaboration:
    """Coordinate candidate generation, expert review, consensus, and tracking."""

    def __init__(self, instance: HumanAICollaborationInstance):
        self.instance = instance
        self.explainable_ai = ExplainableAI(instance)
        expert_configs = [
            ("Supply Chain Strategist", "strategic", {
                "strategic": 0.4, "cost": 0.2, "risk": 0.2, "service": 0.2,
            }),
            ("Regional Operations Manager", "service", {
                "strategic": 0.2, "cost": 0.2, "risk": 0.2, "service": 0.4,
            }),
            ("Financial Analyst", "cost", {
                "strategic": 0.2, "cost": 0.5, "risk": 0.2, "service": 0.1,
            }),
            ("Risk Management Specialist", "risk", {
                "strategic": 0.2, "cost": 0.1, "risk": 0.6, "service": 0.1,
            }),
            ("Market Intelligence Expert", "strategic", {
                "strategic": 0.5, "cost": 0.2, "risk": 0.2, "service": 0.1,
            }),
        ]
        self.human_experts = [
            HumanExpert(name, expertise, preferences)
            for name, expertise, preferences in expert_configs
        ]
        self.collaboration_history = []
        self.decision_outcomes = []
        self.trust_scores = {expert.name: 0.5 for expert in self.human_experts}
        self.ai_confidence_scores = []
        self.human_agreement_rates = []

    def run_collaborative_session(self, session_id: int) -> Dict:
        """Run one complete candidate-generation and consensus session."""
        print(f"\n=== Human–AI Collaborative Session {session_id} ===")
        print(f"Participants: {len(self.human_experts)} expert agents and the AI component")
        session_start = time.perf_counter()

        print("\nPhase 1: Generate candidate networks")
        alternatives = self.explainable_ai.generate_network_alternatives(12)

        print("Phase 2: Collect expert evaluations")
        expert_evaluations = []
        for expert in self.human_experts:
            evaluation = expert.participate_in_discussion(alternatives)
            expert_evaluations.append(evaluation)
            for point in evaluation["discussion_points"]:
                self.explainable_ai.incorporate_human_feedback(
                    point["alternative_id"],
                    point["rating"],
                    point["comments"],
                    expert.name,
                )

        print("Phase 3: Rank alternatives and assess agreement")
        consensus_result = self._build_consensus(alternatives, expert_evaluations)

        print("Phase 4: Record the consensus decision")
        final_decision = self._make_final_decision(consensus_result, alternatives)
        duration = time.perf_counter() - session_start
        session_record = {
            "session_id": session_id,
            "duration": duration,
            "alternatives_generated": len(alternatives),
            "expert_evaluations": expert_evaluations,
            "consensus_result": consensus_result,
            "final_decision": final_decision,
            "ai_confidence": float(np.mean([
                alternative["confidence"] for alternative in alternatives
            ])),
            "human_agreement": self._calculate_agreement_rate(expert_evaluations),
        }
        self._update_trust_scores(session_record)
        session_record["trust_scores"] = self.trust_scores.copy()
        self.collaboration_history.append(session_record)

        print(f"\nSession duration: {duration:.2f} seconds")
        print(f"Selected candidate: {final_decision['chosen_alternative_id']}")
        print(f"Rating agreement indicator: {session_record['human_agreement']:.2f}")
        print(f"Mean AI confidence: {session_record['ai_confidence']:.2f}")
        return session_record

    def _build_consensus(
        self, alternatives: List[Dict], expert_evaluations: List[Dict]
    ) -> Dict:
        """Rank alternatives using trust-weighted ratings plus the AI contribution."""
        alternative_scores = {}
        for alternative in alternatives:
            expert_ratings = []
            weighted_rating_total = 0.0
            for evaluation in expert_evaluations:
                for point in evaluation["discussion_points"]:
                    if point["alternative_id"] == alternative["id"]:
                        trust_weight = self.trust_scores[evaluation["expert"]]
                        weighted_rating_total += point["rating"] * trust_weight
                        expert_ratings.append(point["rating"])

            ai_contribution = alternative["confidence"] * 10 * 0.3
            total_score = weighted_rating_total + ai_contribution
            agreement = (
                max(0.0, 1.0 - float(np.std(expert_ratings)) / 10)
                if expert_ratings
                else 0.5
            )
            alternative_scores[alternative["id"]] = {
                "total_score": total_score,
                "avg_expert_rating": (
                    float(np.mean(expert_ratings)) if expert_ratings else 5.0
                ),
                "agreement_level": agreement,
                "num_expert_ratings": len(expert_ratings),
                "ai_confidence": alternative["confidence"],
                "strategic_alignment": alternative["strategic_alignment"],
            }

        ranked_alternatives = sorted(
            alternative_scores.items(),
            key=lambda item: item[1]["total_score"],
            reverse=True,
        )
        return {
            "ranked_alternatives": ranked_alternatives,
            "top_alternative": ranked_alternatives[0] if ranked_alternatives else None,
            "consensus_strength": self._calculate_consensus_strength(
                alternative_scores
            ),
        }

    def _calculate_consensus_strength(self, alternative_scores: Dict) -> float:
        """Measure the relative gap between the two highest candidate scores."""
        if len(alternative_scores) < 2:
            return 1.0
        scores = sorted(
            (data["total_score"] for data in alternative_scores.values()),
            reverse=True,
        )
        top_score, second_score = scores[:2]
        if top_score <= 0:
            return 0.0
        return float(min(1.0, max(0.0, (top_score - second_score) / top_score)))

    def _make_final_decision(self, consensus_result: Dict, alternatives: List[Dict]) -> Dict:
        """Select the highest-scoring candidate and explain the ranking."""
        top_alternative = consensus_result["top_alternative"]
        if top_alternative is None:
            return {
                "chosen_alternative_id": None,
                "chosen_alternative": None,
                "reasoning": ["No candidate alternatives were generated."],
                "consensus_strength": 0.0,
                "decision_confidence": 0.0,
            }

        alternative_id, score_data = top_alternative
        chosen = next(item for item in alternatives if item["id"] == alternative_id)
        reasoning = [
            f"Candidate {alternative_id} has the highest consensus score.",
            f"Consensus score: {score_data['total_score']:.2f}.",
            f"Expert rating agreement indicator: {score_data['agreement_level']:.2f}.",
            f"AI confidence: {score_data['ai_confidence']:.2f}.",
            f"Strategic alignment proxy: {score_data['strategic_alignment']:.2f}.",
        ]
        if consensus_result["consensus_strength"] > 0.3:
            reasoning.append("The top score is separated from the runner-up.")
        else:
            reasoning.append("The top scores are relatively close; review the ranking.")

        max_score = 10 * len(self.human_experts) + 3
        decision_confidence = min(1.0, score_data["total_score"] / max_score)
        return {
            "chosen_alternative_id": alternative_id,
            "chosen_alternative": chosen,
            "reasoning": reasoning,
            "consensus_strength": consensus_result["consensus_strength"],
            "decision_confidence": decision_confidence,
        }

    def _calculate_agreement_rate(self, expert_evaluations: List[Dict]) -> float:
        """Summarize rating spread for alternatives assessed by multiple experts."""
        ratings_by_alternative = {}
        for evaluation in expert_evaluations:
            for point in evaluation["discussion_points"]:
                ratings_by_alternative.setdefault(point["alternative_id"], []).append(
                    point["rating"]
                )
        spreads = [
            float(np.std(ratings))
            for ratings in ratings_by_alternative.values()
            if len(ratings) >= 2
        ]
        if not spreads:
            return 0.5
        return float(max(0.0, min(1.0, 1.0 - np.mean(spreads) / 5)))

    def _update_trust_scores(self, session_record: Dict) -> None:
        """Update trust indicators using the session agreement and AI confidence."""
        agreement = session_record["human_agreement"]
        ai_confidence = session_record["ai_confidence"]
        if agreement > 0.7 and ai_confidence > 0.7:
            for expert_name in self.trust_scores:
                self.trust_scores[expert_name] = min(
                    1.0, self.trust_scores[expert_name] + 0.05
                )
        elif agreement < 0.5:
            for expert_name in self.trust_scores:
                self.trust_scores[expert_name] = max(
                    0.1, self.trust_scores[expert_name] - 0.02
                )
        self.ai_confidence_scores.append(ai_confidence)
        self.human_agreement_rates.append(agreement)

    def analyze_collaboration_performance(self, num_sessions: int = 10) -> Dict:
        """Run sessions and summarize descriptive collaboration indicators."""
        if num_sessions < 1:
            raise ValueError("At least one session is required.")
        print("\n=== HUMAN–AI COLLABORATION PERFORMANCE ===")
        print(f"Running {num_sessions} sessions...")
        sessions = [
            self.run_collaborative_session(session_id)
            for session_id in range(num_sessions)
        ]
        avg_duration = float(np.mean([session["duration"] for session in sessions]))
        avg_ai_confidence = float(np.mean([
            session["ai_confidence"] for session in sessions
        ]))
        avg_agreement = float(np.mean([
            session["human_agreement"] for session in sessions
        ]))
        initial_trust = 0.5
        final_trust = sessions[-1]["trust_scores"]
        trust_improvement = float(np.mean([
            final_trust[name] - initial_trust for name in final_trust
        ]))
        learning_improvement = self._calculate_learning_improvement()

        print("\nSummary indicators:")
        print(f"  Mean session duration: {avg_duration:.2f} seconds")
        print(f"  Mean AI confidence: {avg_ai_confidence:.3f}")
        print(f"  Mean rating agreement indicator: {avg_agreement:.3f}")
        print(f"  Change in trust indicators: {trust_improvement:.3f}")
        print(f"  Recent feedback-rating trend indicator: {learning_improvement:.3f}")
        return {
            "sessions": sessions,
            "avg_duration": avg_duration,
            "avg_ai_confidence": avg_ai_confidence,
            "avg_agreement": avg_agreement,
            "trust_improvement": trust_improvement,
            "learning_improvement": learning_improvement,
            "trust_evolution": [session["trust_scores"] for session in sessions],
            "confidence_evolution": [session["ai_confidence"] for session in sessions],
            "agreement_evolution": [session["human_agreement"] for session in sessions],
        }

    def _calculate_learning_improvement(self) -> float:
        """Return a normalized recent rating-trend indicator, not a causal estimate."""
        feedback = self.explainable_ai.human_feedback_history
        if len(feedback) < 10:
            return 0.0
        ratings = [item["human_rating"] for item in feedback[-10:]]
        slope = float(np.polyfit(range(len(ratings)), ratings, 1)[0])
        return float(max(0.0, min(1.0, slope / 10)))


print("Human–AI collaboration coordinator is ready.")

## 5. Run the simulation

This cell executes ten sessions using the same instance and collaboration system. Candidate layouts are reproducible for a given alternative identifier, while expert rating noise and measured run times can vary with the sequence of execution. The summaries are descriptive outputs of the simulation, not evidence of operational performance.

In [ ]:
collaboration_system = HumanAICollaboration(human_ai_instance)
performance_results = collaboration_system.analyze_collaboration_performance(
    num_sessions=10
)

print("\n=== SUMMARY OF COLLABORATION RESULTS ===")
print(f"Sessions analyzed: {len(performance_results['sessions'])}")
print(f"Mean AI confidence: {performance_results['avg_ai_confidence']:.3f}")
print(f"Mean rating agreement indicator: {performance_results['avg_agreement']:.3f}")
print(f"Change in trust indicators: {performance_results['trust_improvement']:.3f}")
print(
    "Recent feedback-rating trend indicator: "
    f"{performance_results['learning_improvement']:.3f}"
)

print("\nTrust indicators after the final session:")
for expert_name, score in performance_results["trust_evolution"][-1].items():
    print(f"  {expert_name}: {score:.3f}")

last_session = performance_results["sessions"][-1]
chosen_alternative = last_session["final_decision"]["chosen_alternative"]
if chosen_alternative is not None:
    metrics = chosen_alternative["metrics"]
    print("\nSelected candidate in the final session:")
    print(f"  Candidate identifier: {chosen_alternative['id']}")
    print(f"  Expected cost: {metrics['expected_cost']:.2f} cost units")
    print(f"  Service proxy: {metrics['service_level']:.3f}")
    print(f"  AI confidence: {chosen_alternative['confidence']:.3f}")
    print(
        "  Ranking rationale: "
        + "; ".join(last_session["final_decision"]["reasoning"][:3])
    )

# Check that every customer has one assignment to an open facility.
for session in performance_results["sessions"]:
    for alternative in session["consensus_result"]["ranked_alternatives"]:
        alternative_id = alternative[0]
        candidate = next(
            item
            for item in collaboration_system.explainable_ai.generate_network_alternatives(12)
            if item["id"] == alternative_id
        )
        assignments = candidate["configuration"]["customer_assignments"]
        assert set(assignments) == set(human_ai_instance.customers)
        assert set(assignments.values()).issubset(
            candidate["configuration"]["open_facilities"]
        )
        break

## 6. Visualize collaboration indicators

The panels show trust indicators, mean AI confidence, rating agreement, candidate decision indicators, and session duration. Trust values in each session record are captured after that session's trust update. Session duration is included for workflow context; it depends on the runtime environment and is not a measure of decision quality.

In [ ]:
def visualize_human_ai_collaboration(
    performance_results: Dict, collaboration_system: HumanAICollaboration
) -> None:
    """Plot descriptive indicators from the recorded collaboration sessions."""
    sessions = performance_results["sessions"]
    if not sessions:
        raise ValueError("At least one recorded session is required for plotting.")
    session_numbers = list(range(1, len(sessions) + 1))
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Human–AI Collaboration: Simulated Decision Indicators",
        fontsize=16,
        fontweight="bold",
    )

    ax = axes[0, 0]
    trust_evolution = performance_results["trust_evolution"]
    for expert in collaboration_system.human_experts:
        values = [record[expert.name] for record in trust_evolution]
        ax.plot(session_numbers, values, marker="o", linewidth=1.8, label=expert.name)
    ax.set_xlabel("Session number")
    ax.set_ylabel("Trust indicator (0–1)")
    ax.set_title("Trust indicators after each session")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[0, 1]
    ax.plot(
        session_numbers,
        performance_results["confidence_evolution"],
        "o-",
        label="Mean AI confidence",
    )
    ax.plot(
        session_numbers,
        performance_results["agreement_evolution"],
        "s-",
        label="Rating agreement indicator",
    )
    ax.set_xlabel("Session number")
    ax.set_ylabel("Indicator (0–1)")
    ax.set_title("Confidence and rating agreement")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[1, 0]
    decisions = [session["final_decision"] for session in sessions]
    decision_confidence = [item["decision_confidence"] for item in decisions]
    consensus_strength = [item["consensus_strength"] for item in decisions]
    ax.plot(session_numbers, decision_confidence, "o-", label="Decision-score indicator")
    ax.plot(session_numbers, consensus_strength, "s-", label="Top-score gap")
    ax.set_xlabel("Session number")
    ax.set_ylabel("Indicator (0–1)")
    ax.set_title("Decision score and consensus gap")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[1, 1]
    durations = [session["duration"] for session in sessions]
    alternatives_generated = [session["alternatives_generated"] for session in sessions]
    ax.plot(session_numbers, durations, "o-", color="brown", label="Duration (seconds)")
    ax.set_xlabel("Session number")
    ax.set_ylabel("Duration (seconds)", color="brown")
    ax.set_title("Session runtime and candidate count")
    ax.grid(True, alpha=0.3)
    ax_right = ax.twinx()
    ax_right.plot(
        session_numbers,
        alternatives_generated,
        "s--",
        color="teal",
        label="Candidates generated",
    )
    ax_right.set_ylabel("Number of candidates", color="teal")
    lines = ax.get_lines() + ax_right.get_lines()
    ax.legend(lines, [line.get_label() for line in lines], loc="upper left")

    fig.tight_layout()
    plt.show()


visualize_human_ai_collaboration(performance_results, collaboration_system)

## 7. Inspect explanations and feedback

This review examines a small set of generated alternatives and summarizes recorded expert ratings. The confidence score is produced by a simple rule with a small, seeded perturbation; it is not a statistically calibrated probability. Preference weights show the state maintained by the feedback routine, but—as noted above—those weights do not alter the candidate ranking in this implementation.

In [ ]:
def analyze_explainable_ai_performance(
    collaboration_system: HumanAICollaboration,
) -> Dict:
    """Display sample explanations and summarize accumulated human feedback."""
    print("\n=== EXPLANATION AND FEEDBACK REVIEW ===")
    sample_alternatives = (
        collaboration_system.explainable_ai.generate_network_alternatives(5)
    )
    for alternative in sample_alternatives:
        metrics = alternative["metrics"]
        explanation = alternative["explanation"]
        print(f"\nCandidate {alternative['id']}:")
        print(f"  AI confidence: {alternative['confidence']:.3f}")
        print(f"  Strategic alignment proxy: {alternative['strategic_alignment']:.3f}")
        print(f"  Expected cost: {metrics['expected_cost']:.2f} cost units")
        print(f"  Service proxy: {metrics['service_level']:.3f}")
        print(f"  Illustrative risk score: {metrics['risk_score']:.3f}")
        print(f"  Explanation: {explanation['summary']}")
        for section in ("key_decisions", "trade_offs", "recommendations"):
            if explanation[section]:
                print(f"  {section.replace('_', ' ').title()}: {explanation[section][0]}")

    feedback_history = (
        collaboration_system.explainable_ai.human_feedback_history
    )
    print(f"\nRecorded feedback entries: {len(feedback_history)}")
    expert_feedback = {}
    avg_rating = 0.0
    if feedback_history:
        ratings = [item["human_rating"] for item in feedback_history]
        avg_rating = float(np.mean(ratings))
        rating_std = float(np.std(ratings))
        print(f"Mean simulated rating: {avg_rating:.2f} (standard deviation {rating_std:.2f})")
        for item in feedback_history:
            expert_feedback.setdefault(item["expert_focus"], []).append(
                item["human_rating"]
            )
        print("Ratings by expert:")
        for expert_name, expert_ratings in expert_feedback.items():
            print(
                f"  {expert_name}: mean {np.mean(expert_ratings):.2f}, "
                f"standard deviation {np.std(expert_ratings):.2f}, "
                f"n={len(expert_ratings)}"
            )

    current_weights = collaboration_system.explainable_ai.preference_weights.copy()
    print("\nStored preference weights:")
    for category, weight in current_weights.items():
        print(f"  {category}: {weight:.3f}")
    assert np.isclose(sum(current_weights.values()), 1.0)
    return {
        "sample_alternatives": sample_alternatives,
        "feedback_history": feedback_history,
        "current_weights": current_weights,
        "avg_human_rating": avg_rating,
        "expert_feedback": expert_feedback,
    }


xai_analysis = analyze_explainable_ai_performance(collaboration_system)

## 8. Interpret the indicators cautiously

The comparison below places simulation-derived indicators beside illustrative reference values retained as assumptions in the example. These reference values are not measured results, validated baselines, or evidence that one approach outperforms another. Decision quality, strategic alignment, adaptability, and transparency are represented by simple proxies, so the table and plots are useful for discussing indicator construction rather than for making a causal business comparison. Runtime is converted from seconds to hours for display, but it is the execution time of a small simulation and should not be interpreted as the duration of a real strategic planning process.

In [ ]:
def compare_human_ai_vs_traditional(performance_results: Dict) -> Dict:
    """Display simulated indicators beside explicitly illustrative reference values."""
    print("\n=== SIMULATION INDICATORS AND ILLUSTRATIVE REFERENCES ===")
    print("Reference values are assumptions for discussion, not empirical benchmarks.")
    reference_metrics = {
        "decision_quality": 0.65,
        "decision_time_hours": 48.0,
        "agreement": 0.55,
        "strategic_alignment": 0.70,
        "adaptability": 0.30,
        "transparency": 0.40,
    }
    simulated = {
        "decision_quality": (
            performance_results["avg_agreement"]
            * performance_results["avg_ai_confidence"]
        ),
        "decision_time_hours": performance_results["avg_duration"] / 3600,
        "agreement": performance_results["avg_agreement"],
        "strategic_alignment": float(np.mean([
            session["final_decision"]["chosen_alternative"]["strategic_alignment"]
            for session in performance_results["sessions"]
            if session["final_decision"]["chosen_alternative"] is not None
        ])),
        "adaptability": performance_results["learning_improvement"],
        "transparency": 0.85,
    }
    print(f"\n{'Indicator':<26} {'Simulation':>14} {'Reference assumption':>22}")
    print("-" * 64)
    for key, label in [
        ("decision_quality", "Decision-quality proxy"),
        ("decision_time_hours", "Runtime (hours)"),
        ("agreement", "Rating agreement"),
        ("strategic_alignment", "Strategic-alignment proxy"),
        ("adaptability", "Feedback-trend indicator"),
        ("transparency", "Transparency assumption"),
    ]:
        print(f"{label:<26} {simulated[key]:>14.3f} {reference_metrics[key]:>22.3f}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    labels = ["Simulation", "Reference assumption"]
    colors = ["#4ECDC4", "#FF6B6B"]

    values = [simulated["decision_quality"], reference_metrics["decision_quality"]]
    bars = axes[0, 0].bar(labels, values, color=colors, alpha=0.85)
    axes[0, 0].set_ylabel("Score (0–1)")
    axes[0, 0].set_title("Decision-quality proxy")
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        axes[0, 0].text(bar.get_x() + bar.get_width() / 2, value + 0.02,
                        f"{value:.3f}", ha="center")

    values = [simulated["decision_time_hours"], reference_metrics["decision_time_hours"]]
    bars = axes[0, 1].bar(labels, values, color=colors, alpha=0.85)
    axes[0, 1].set_ylabel("Hours")
    axes[0, 1].set_title("Simulation runtime and reference assumption")
    axes[0, 1].grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        axes[0, 1].text(bar.get_x() + bar.get_width() / 2, value,
                        f"{value:.3f}", ha="center", va="bottom")

    capability_labels = ["Quality", "Alignment", "Adaptability", "Transparency"]
    keys = ["decision_quality", "strategic_alignment", "adaptability", "transparency"]
    x = np.arange(len(keys))
    width = 0.36
    axes[1, 0].bar(
        x - width / 2, [simulated[key] for key in keys], width,
        label="Simulation indicator", color=colors[0], alpha=0.85,
    )
    axes[1, 0].bar(
        x + width / 2, [reference_metrics[key] for key in keys], width,
        label="Reference assumption", color=colors[1], alpha=0.85,
    )
    axes[1, 0].set_xlabel("Indicator")
    axes[1, 0].set_ylabel("Score (0–1)")
    axes[1, 0].set_title("Illustrative indicator comparison")
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(capability_labels)
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].grid(True, axis="y", alpha=0.3)
    axes[1, 0].legend()

    trust_values = [
        float(np.mean(list(record.values())))
        for record in performance_results["trust_evolution"]
    ]
    session_numbers = list(range(1, len(trust_values) + 1))
    axes[1, 1].plot(session_numbers, trust_values, "o-", label="Mean trust indicator")
    axes[1, 1].axhline(0.5, color="red", linestyle="--", label="Initial value")
    axes[1, 1].set_xlabel("Session number")
    axes[1, 1].set_ylabel("Trust indicator (0–1)")
    axes[1, 1].set_title("Trust indicators across sessions")
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()

    fig.tight_layout()
    plt.show()
    return {"simulated_indicators": simulated, "reference_assumptions": reference_metrics}


comparison_results = compare_human_ai_vs_traditional(performance_results)

## 9. Key takeaways and limitations

This example illustrates a human–AI decision workflow: candidate generation, explanation, selective expert review, a trust-weighted consensus ranking, and recording of feedback. Its key educational value is the connection between the workflow and the quantities used in the code—not a claim that the simulated indicators establish business value.

- **Explanations support review.** Candidate choices and calculated indicators are exposed so that a reviewer can inspect the rationale. The explanations are rule-based summaries, not a guarantee that a recommendation is valid.
- **Consensus depends on the scoring rule.** Trust-weighted ratings and the AI-confidence contribution determine the ranking. Candidates that experts do not review receive no expert-rating contribution, so the ranking should be interpreted with that selection rule in mind.
- **The feedback state is limited.** Expert feedback updates preference weights, but those weights do not affect recommendations or consensus in this implementation. The recent rating-trend indicator is descriptive and must not be interpreted as measured learning improvement.
- **Capacity is a review indicator, not a feasibility constraint.** Candidate generation does not enforce capacity. A candidate with expected utilization above one requires further review and cannot be treated as a feasible plan based on this notebook alone.
- **The metrics are illustrative.** The service, risk, strategic-alignment, confidence, trust, and decision-quality indicators use simplified rules and unvalidated assumptions. The comparison references are assumptions, not measured results or a controlled comparison.

A real application would require validated demand and cost data, explicit operational constraints, carefully defined performance measures, accountable human review, and evaluation against an appropriate baseline. Those additions are outside the scope of this simulation.